# Блокнот 4. Логистическая регрессия и метрики классификации

Курс «Машинное обучение с нуля», модуль 4. Разделы: 1 — урок 4.1, 2–3 — уроки 4.2–4.3, 4 — урок 4.4, 5–7 — уроки 4.5–4.7. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение — следующая ячейка с `assert` проверит его.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(0)

## 1. Формула Байеса на частотах (урок 4.1)

1 % дроидов неисправен. Тест ловит неисправного в 90 % случаев, а у исправного ошибочно бьёт тревогу в 9 % случаев. Смоделируем 100 000 дроидов и сравним частоту с формулой.

In [ ]:
n = 100_000
broken = rng.random(n) < 0.01
alarm = np.where(broken, rng.random(n) < 0.90, rng.random(n) < 0.09)
print("доля неисправных среди тех, на кого тест показал:", round(broken[alarm].mean(), 3))

**Задание 1.** Посчитайте ту же вероятность по формуле Байеса: $P(A \mid B) = \frac{P(B \mid A) P(A)}{P(B \mid A) P(A) + P(B \mid \bar{A}) P(\bar{A})}$.

In [ ]:
posterior = ...  # ваш код
round(posterior, 4)

In [ ]:
assert abs(posterior - 0.0917) < 1e-3, f"получилось {posterior}"
print("Верно! Положительный тест — это всего ~9 % вероятности поломки.")

## 2. Логистическая регрессия своими руками (уроки 4.2–4.3)

Два класса точек на плоскости. Обучим модель градиентным спуском по log-loss. Градиент log-loss выглядит так же, как у MSE, только вместо прогноза — вероятность: $\frac{\partial L}{\partial w} = \frac{1}{n} X^T (p - y)$.

In [ ]:
X = np.r_[rng.normal([2, 2], 1, (100, 2)), rng.normal([5, 4], 1, (100, 2))]
y = np.r_[np.zeros(100), np.ones(100)]


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def log_loss(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

**Задание 2.** Допишите шаг обучения: вероятности `p = sigmoid(X @ w + b)`, градиенты `grad_w = X.T @ (p - y) / n`, `grad_b = (p - y).mean()` и шаг против градиента.

In [ ]:
def fit_logreg(X, y, lr=0.1, steps=2000):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(steps):
        p = ...  # ваш код
        w = ...  # ваш код
        b = ...  # ваш код
    return w, b


w, b = fit_logreg(X, y)
p = sigmoid(X @ w + b)
print("веса", w.round(2), "b", round(b, 2), "log-loss", round(log_loss(y, p), 3), "accuracy", ((p >= 0.5) == y).mean())

In [ ]:
from sklearn.linear_model import LogisticRegression

sk = LogisticRegression(C=1e6).fit(X, y)  # C огромное — почти без регуляризации
assert ((p >= 0.5) == y).mean() > 0.9, "модель должна угадывать больше 90 %"
print("Верно! scikit-learn:", sk.coef_.round(2), sk.intercept_.round(2))

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1, 8, 200), np.linspace(-1, 7, 200))
pp = sigmoid(np.c_[xx.ravel(), yy.ravel()] @ w + b).reshape(xx.shape)
plt.contourf(xx, yy, pp, levels=10, cmap="RdYlBu_r", alpha=0.4)
plt.contour(xx, yy, pp, levels=[0.5], colors="black")
plt.scatter(*X.T, c=y, cmap="coolwarm", s=10)
plt.title("Вероятность класса 1 и граница p = 0,5")
plt.show()

## 3. Softmax (урок 4.4)

**Задание 3.** Напишите устойчивый softmax: вычтите максимум перед экспонентой, иначе `np.exp(1000)` переполнится.

In [ ]:
def softmax(z):
    return ...  # ваш код

In [ ]:
assert np.allclose(softmax(np.array([2.0, 1.0, 0.1, -1.0])), [0.638, 0.235, 0.095, 0.032], atol=1e-3)
assert np.allclose(softmax(np.array([1000.0, 1001.0])), [0.269, 0.731], atol=1e-3), "переполнение: вычтите максимум"
print("Верно!")

Многоклассовая логистическая регрессия на пингвинах (урок 2.4): три вида, четыре числовых признака.

In [ ]:
import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

peng = sns.load_dataset("penguins").dropna()
Xp = peng[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
model_p = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
print("accuracy по 5 фолдам:", cross_val_score(model_p, Xp, peng["species"], cv=5).mean().round(3))
model_p.fit(Xp, peng["species"])
print(model_p.classes_, model_p.predict_proba(Xp.iloc[:3]).round(3))

## 4. Опухоли молочной железы: метрики под капотом (уроки 4.5–4.7)

569 опухолей, 30 признаков клеточных ядер. **Внимание:** в scikit-learn метка 0 — злокачественная, 1 — доброкачественная. Перекодируем, чтобы класс 1 означал «злокачественная» — именно её важно не пропустить.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
yb = (data.target == 0).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(data.data, yb, test_size=0.25, stratify=yb, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
print(confusion_matrix(y_te, proba >= 0.5))

**Задание 4.** Посчитайте precision и recall для класса «злокачественная» **вручную** из TP, FP, FN при пороге 0,5 и сравните с `precision_score` и `recall_score`.

In [ ]:
pred = (proba >= 0.5).astype(int)
tp = np.sum((y_te == 1) & (pred == 1))
fp = np.sum((y_te == 0) & (pred == 1))
fn = np.sum((y_te == 1) & (pred == 0))
precision = ...  # ваш код
recall = ...  # ваш код
print(round(precision, 3), round(recall, 3))

In [ ]:
assert np.isclose(precision, precision_score(y_te, pred)) and np.isclose(recall, recall_score(y_te, pred))
print("Верно! Пропущено злокачественных:", fn)

**Задание 5.** Порог выбирают **не по тесту**, а по предсказаниям кросс-валидации на обучающей части (`cross_val_predict`). Найдите самый высокий порог из списка, при котором recall на обучающей части ≥ 0,98. Затем — один раз — проверьте его на тесте.

In [ ]:
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_tr, y_tr, cv=5, method="predict_proba")[:, 1]
for t in (0.5, 0.3, 0.2, 0.1, 0.05):
    pr = (oof >= t).astype(int)
    print(f"порог {t}: recall {recall_score(y_tr, pr):.3f}, precision {precision_score(y_tr, pr):.3f}")
best_threshold = ...  # ваш код

In [ ]:
assert best_threshold == 0.2, f"получилось {best_threshold}"
test_pred = (proba >= best_threshold).astype(int)
print("Порог", best_threshold, "на тесте: recall", round(recall_score(y_te, test_pred), 3),
      "precision", round(precision_score(y_te, test_pred), 3), "матрица", confusion_matrix(y_te, test_pred).ravel())
print("ROC-AUC на тесте", round(roc_auc_score(y_te, proba), 3))

## 5. ROC и PR-кривые, калибровка

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.naive_bayes import GaussianNB

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_te, proba, ax=axes[0])
PrecisionRecallDisplay.from_predictions(y_te, proba, ax=axes[1])
nb = GaussianNB().fit(X_tr, y_tr)
CalibrationDisplay.from_predictions(y_te, proba, n_bins=6, strategy="quantile", ax=axes[2], name="логистическая")
CalibrationDisplay.from_predictions(y_te, nb.predict_proba(X_te)[:, 1], n_bins=6, strategy="quantile", ax=axes[2], name="наивный Байес")
plt.tight_layout()
plt.show()